**Overview:**

This notebook automates the extraction of road network data for various designated places defined in shapefiles. The workflow seamlessly integrates data loading, processing, and output generation for both US and Canadian datasets, and is structured as follows:

1. **Configuration & Setup:**
   - **Global Parameters:**  
     Key parameters are defined, including the buffer size (in meters) to expand each place’s polygon, the number of parallel workers, and paths for both input shapefiles and output directories.
   - **Directory & Logging Setup:**  
     The script ensures that the designated output directory exists and configures a logging mechanism to capture progress, errors, and task summaries throughout execution.

2. **Input Data Loading:**
   - **US Shapefiles:**  
     The script scans the input directory for state folders and loads US designated places shapefiles. An optional state filter (`DOWNLOAD_ONLY_STATES`) allows processing only specified states.
   - **Canadian Shapefile:**  
     A dedicated function loads the Canadian designated places shapefile. The function also renames and creates necessary columns (such as `NAME` and `GEOID`) to ensure consistency.

3. **Processing Each Designated Place:**
   - **Place-Level Processing:**  
     For each place (each row in the GeoDataFrame), the following steps are performed:
     - **Data Extraction:**  
       Relevant attributes (e.g., place name, unique identifier) and geometry are extracted from the dataset.
     - **Buffering:**  
       A spatial buffer is applied to the place’s polygon to capture an extended area. The process involves converting to a suitable projected coordinate system (UTM), applying the buffer, and reprojecting back to EPSG:4326.
     - **Road Network Query:**  
       The buffered polygon is used to request the road network from OpenStreetMap using OSMnx. The request is configured to extract driving networks and includes options to simplify the network geometry.
     - **Saving the Network:**  
       The extracted road network is saved as a GraphML file in a structured output folder organized by country, region (state or province), and place (using a sanitized filename with its unique identifier).

4. **Parallel Processing & Task Management:**
   - **Concurrent Execution:**  
     The script uses a `ProcessPoolExecutor` to process multiple designated places concurrently. This parallel approach significantly reduces the total processing time.
   - **Progress Tracking & Logging:**  
     A progress bar (via `tqdm`) visualizes the processing of places, while logging captures detailed information on each task's success or failure. At the end of processing, a summary of the number of successfully processed and failed places is logged.

5. **Main Execution Flow:**
   - The `main()` function orchestrates the entire workflow:
     - It first processes all relevant US shapefiles, loading and processing each state's designated places.
     - It then processes the Canadian shapefile, handling province-level subdivision if applicable or placing all features under a general Canada folder.
     - Comprehensive logging throughout the process ensures that issues are flagged and that the overall execution is clearly documented.

In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

# %%
# =============================================================================
# ============================ IMPORTS & CONFIGURATION ========================
# =============================================================================

import os
import geopandas as gpd
import osmnx as ox
from tqdm import tqdm
from time import sleep
import concurrent.futures
import logging
import gc

# 🏷️ Toggle for output location:
USE_LOCAL_OUTPUT = False  # True = local dir, False = network share

# 📁 Define local vs. network share paths
LOCAL_OUTPUT_DIR  = os.path.expanduser('~/osmnx_outputs')
REMOTE_OUTPUT_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'osmnx_road_network', '2024')

# 🌐 Effective output directory
OUTPUT_DIR = LOCAL_OUTPUT_DIR if USE_LOCAL_OUTPUT else REMOTE_OUTPUT_DIR

# Other config
INPUT_DIR_US         = os.path.join(WRI_PROJECT_ROOT, 'data', 'multi_domain_data', 'raw', 'boundary_layers', 'us_census_designated_places', '2024')
INPUT_PATH_CANADA    = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate', 'road_network', '2024', 'canada_designated_places_equivalent', 'canada_cdp_equivalent.shp')
BUFFER_SIZE_METERS   = 2000    
NUM_WORKERS          = 50 
DOWNLOAD_ONLY_STATES = ['New_Mexico']  # If blank, all states will be downloaded
PROCESS_CANADA       = True  # 🍁 True = process Canada, False = skip Canada
OVERWRITE_EXISTING   = True
RUN_IN_PARALLEL      = True  # 🔀

# Ensure output & log directories exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_DIR = os.path.expanduser("~/logs")
os.makedirs(LOG_DIR, exist_ok=True)

# Configure logging
logging.basicConfig(
    filename=os.path.join(LOG_DIR, 'graph_download.log'),
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

# %%
# =============================================================================
# ============================ HELPER FUNCTIONS ================================
# =============================================================================

def sanitize_filename(name):
    """🛡️ Make filenames safe by keeping only letters & numbers."""
    return "".join(c if c.isalnum() else "_" for c in name).strip("_")

def load_us_shapefiles():
    """
    🔍 Find US shapefiles, properly extracting state names that may contain underscores.
    Handles cases like 'New_Mexico_35' -> 'New_Mexico'
    """
    shapefiles = []
    for state_dir in os.listdir(INPUT_DIR_US):
        path = os.path.join(INPUT_DIR_US, state_dir)
        if not os.path.isdir(path):
            continue
        
        # Extract state name by removing the FIPS code suffix
        # Assumes format: StateName_FIPSCode (e.g., 'New_Mexico_35')
        parts = state_dir.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            # Last part is FIPS code, join all parts except the last
            state_name = '_'.join(parts[:-1])
        else:
            # Fallback: use the first part only
            state_name = parts[0]
        
        # Debug output to see what state names are being extracted
        print(f"Directory: {state_dir} -> State name: {state_name}")
        
        # Check if this state should be processed
        if DOWNLOAD_ONLY_STATES and state_name not in DOWNLOAD_ONLY_STATES:
            continue
            
        # Find shapefile in the directory
        for file in os.listdir(path):
            if file.endswith('.shp'):
                shapefiles.append((os.path.join(path, file), state_name))
    return shapefiles

def load_canadian_shapefile():
    """
    🍁 Load & normalize the Canada designated places shapefile.
    """
    try:
        gdf = gpd.read_file(INPUT_PATH_CANADA)

        rename = {}
        if 'Name' in gdf:    rename['Name'] = 'NAME'
        if 'GID' in gdf:     rename['GID']  = 'GEOID'
        if 'DPLUID' in gdf and 'GEOID' not in gdf:
            rename['DPLUID'] = 'GEOID'
        if rename:
            gdf = gdf.rename(columns=rename)

        if 'NAME' not in gdf:
            gdf['NAME'] = [f"Place_{i}" for i in range(len(gdf))]
        if 'GEOID' not in gdf:
            gdf['GEOID'] = [f"CAN_{i}" for i in range(len(gdf))]

        return gdf

    except Exception as e:
        msg = f"❌ [load_canadian_shapefile] Error: {e}"
        print(msg)  # only print on error 🚨
        logging.error(msg)
        return None

# %%
# =============================================================================
# ========================= PROCESS SINGLE PLACE ==============================
# =============================================================================

def process_place(place, region_name, country='US'):
    """
    🚗 Download and save road network for one place,
    with correct CRS buffering and memory cleanup.
    """
    try:
        place_name = place.get('NAME', 'Unknown')
        geoid      = place.get('GEOID', str(place.name))

        sanitized_name   = sanitize_filename(place_name)
        folder_name      = f"{sanitized_name}_{geoid}"
        place_output_dir = os.path.join(OUTPUT_DIR, country, region_name, folder_name)
        os.makedirs(place_output_dir, exist_ok=True)

        graph_fp = os.path.join(place_output_dir, f"{sanitized_name}_{geoid}.graphml")
        if os.path.exists(graph_fp) and not OVERWRITE_EXISTING:
            return  # skip silently

        # ── CRS buffering ────────────────────────────────────────────────────────
        src = gpd.GeoSeries([place.geometry], crs='EPSG:4326')
        proj = src.to_crs(epsg=5070)
        buffered = proj.buffer(BUFFER_SIZE_METERS).iloc[0]
        buffered_wgs84 = gpd.GeoSeries([buffered], crs=5070).to_crs(epsg=4326).iloc[0]

        # ── Download & save ────────────────────────────────────────────────────
        G = ox.graph_from_polygon(
            buffered_wgs84,
            network_type='drive',
            retain_all=True,
            simplify=True
        )
        ox.save_graphml(G, graph_fp)

        # ── Memory cleanup ─────────────────────────────────────────────────────
        del G
        gc.collect()

    except Exception as e:
        err = f"❌ Error for {place_name} ({geoid}): {e}"
        print(err)  # only print on error 🚨
        logging.error(err)
        return err

# %%
# =============================================================================
# ====================== PROCESS PLACES GEODATAFRAME ==========================
# =============================================================================

def process_geodataframe(gdf_places, region_name, country='US'):
    """
    🏃‍♂️ Iterate over places, either in parallel or sequentially,
    using a tqdm progress bar and printing only on errors.
    """
    try:
        # Ensure CRS is WGS84
        if gdf_places.crs is None:
            gdf_places.set_crs(epsg=4326, inplace=True)
        else:
            gdf_places = gdf_places.to_crs(epsg=4326)

        # Print the output folder where files are being saved
        state_output_dir = os.path.join(OUTPUT_DIR, country, region_name)
        print(f"📁 Saving files to: {state_output_dir}")

        places = list(gdf_places.iterrows())
        failed = 0

        if RUN_IN_PARALLEL:
            with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
                futures = {
                    executor.submit(process_place, row, region_name, country): row.get('NAME','Unknown')
                    for _, row in places
                }
                for future in tqdm(concurrent.futures.as_completed(futures),
                                   total=len(places),
                                   desc=f"{country}-{region_name}"):
                    err = future.result()
                    if err:
                        failed += 1
        else:
            for _, row in tqdm(places, total=len(places), desc=f"{country}-{region_name}"):
                err = process_place(row, region_name, country)
                if err:
                    failed += 1

        # log summary (no print)
        logging.info(f"{country}-{region_name}: {len(places)-failed} succeeded, {failed} failed")

    except Exception as e:
        msg = f"❌ Error for {country}-{region_name}: {e}"
        print(msg)  # only print on critical error 🚨
        logging.error(msg)

# %%
# =============================================================================
# ============================= MAIN EXECUTION ================================
# =============================================================================

def main():
    # --- U.S. Places ---
    us_files = load_us_shapefiles()
    for shp, state in us_files:
        # 📂 Print the state name and shapefile path before processing
        print(f"▶️ Processing US state '{state}': {shp}")
        try:
            gdf = gpd.read_file(shp)
            process_geodataframe(gdf, state, country='US')
        except Exception as e:
            msg = f"❌ Error reading US shapefile {shp}: {e}"
            print(msg)
            logging.error(msg)

    # --- Canada Places ---
    if PROCESS_CANADA:
        can = load_canadian_shapefile()
        if can is not None:
            if 'Province' in can:
                for prov in can['Province'].unique():
                    try:
                        dfp = can[can['Province'] == prov]
                        process_geodataframe(dfp, prov, country='Canada')
                    except Exception as e:
                        msg = f"❌ Error for Canada province {prov}: {e}"
                        print(msg)
                        logging.error(msg)
            else:
                process_geodataframe(can, 'Canada', country='Canada')
    else:
        logging.info("Skipping Canada processing (PROCESS_CANADA = False)")

    logging.info("Main execution complete")

if __name__ == "__main__":
    main()